In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(__file__).parent if '__file__' in dir() else Path().resolve()

df = pd.read_csv(
    BASE / "../../../data/processed/2025/datos_limpios.csv",
    low_memory=False
)

# 1. Definir pozos eficientes vs ineficientes
print("=== CLASIFICACIÓN DE POZOS ===")
por_pozo = df.groupby(['idpozo', 'empresa', 'provincia', 'cuenca', 
                        'tipo_de_recurso', 'tipoextraccion', 'formacion'])[
    ['prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'tef']
].sum().reset_index()

por_pozo = por_pozo[por_pozo['prod_pet'] > 0].copy()

por_pozo['productividad'] = por_pozo.apply(
    lambda x: x['prod_pet'] / x['tef'] if x['tef'] > 0 else 0, axis=1
)

por_pozo['rendimiento'] = pd.qcut(
    por_pozo['productividad'], 
    q=3, 
    labels=['Bajo', 'Medio', 'Alto']
)

print(por_pozo['rendimiento'].value_counts())

# 2. Patrones en pozos de alto rendimiento
print("\n=== PATRONES EN POZOS DE ALTO RENDIMIENTO ===")
alto_rendimiento = por_pozo[por_pozo['rendimiento'] == 'Alto']

print("\nDistribución por tipo de recurso:")
print(alto_rendimiento['tipo_de_recurso'].value_counts())

print("\nDistribución por provincia:")
print(alto_rendimiento['provincia'].value_counts())

print("\nDistribución por cuenca:")
print(alto_rendimiento['cuenca'].value_counts())

print("\nDistribución por tipo de extracción:")
print(alto_rendimiento['tipoextraccion'].value_counts())

print("\nTop 10 formaciones más productivas:")
print(alto_rendimiento['formacion'].value_counts().head(10))

# 3. Patrones en pozos de bajo rendimiento
print("\n=== PATRONES EN POZOS DE BAJO RENDIMIENTO ===")
bajo_rendimiento = por_pozo[por_pozo['rendimiento'] == 'Bajo']

print("\nDistribución por tipo de recurso:")
print(bajo_rendimiento['tipo_de_recurso'].value_counts())

print("\nDistribución por provincia:")
print(bajo_rendimiento['provincia'].value_counts())

print("\nDistribución por cuenca:")
print(bajo_rendimiento['cuenca'].value_counts())

print("\nDistribución por tipo de extracción:")
print(bajo_rendimiento['tipoextraccion'].value_counts())

# 4. Comparación directa alto vs bajo rendimiento
print("\n=== COMPARACIÓN ALTO VS BAJO RENDIMIENTO ===")
comparacion = por_pozo.groupby('rendimiento').agg(
    cantidad_pozos=('idpozo', 'count'),
    prod_pet_promedio=('prod_pet', 'mean'),
    productividad_promedio=('productividad', 'mean'),
    iny_agua_promedio=('iny_agua', 'mean')
).round(2)
print(comparacion)

# 5. Guardar para Power BI
por_pozo.to_csv(
    BASE / "../../../data/processed/2025/patrones_pozos.csv",
    index=False
)
print("\nArchivo guardado correctamente")

=== CLASIFICACIÓN DE POZOS ===
rendimiento
Bajo     9525
Alto     9525
Medio    9524
Name: count, dtype: int64

=== PATRONES EN POZOS DE ALTO RENDIMIENTO ===

Distribución por tipo de recurso:
tipo_de_recurso
CONVENCIONAL       7095
NO CONVENCIONAL    2429
NO DISCRIMINADO       1
Name: count, dtype: int64

Distribución por provincia:
provincia
Chubut              3012
Neuquén             2928
Mendoza             1628
Santa Cruz          1131
Rio Negro            430
La Pampa             255
Tierra del Fuego      68
Salta                 34
Estado Nacional       24
Formosa               11
Jujuy                  4
Name: count, dtype: int64

Distribución por cuenca:
cuenca
NEUQUINA           4454
GOLFO SAN JORGE    4060
CUYANA              787
AUSTRAL             175
NOROESTE             49
Name: count, dtype: int64

Distribución por tipo de extracción:
tipoextraccion
Bombeo Mecánico              3471
Electrosumergible            2172
Surgencia Natural            1866
Cavidad Progresiva 

C:\Users\santi\AppData\Local\Temp\ipykernel_8564\401772814.py:69: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  comparacion = por_pozo.groupby('rendimiento').agg(



Archivo guardado correctamente
